In [ ]:
#############################################
###  Phase 2. Calculate the Heritability  ###
#############################################
# Code for LLM from https://github.com/xbeat/Machine-Learning/blob/main/Linear%20Mixed%20Models%20in%20Python.md

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter('ignore', ConvergenceWarning)


base_dir = './'
input_data_path = './HarvestData.xlsx'

df_raw = pd.read_excel(input_data_path)
traits = df_raw.columns[5:] 
treatments = ['Trt1', 'Trt2']

h2_results = []

# Set the threshold of H^2 (min = 0.3)
H2_THRESHOLD = 0.30

# Calculate the Heritability and run LLM
print("Phase 2: Calculating Heritability (H^2)...")

for trait in traits:
    trait_h2_info = {'Trait': trait}
    pass_flag = True
    
    for trt in treatments:
        df_subset = df_raw[df_raw['Treatment'] == trt][['Genotype', trait]].dropna().copy()
        df_subset.columns = ['Genotype', 'Value']
        
        if len(df_subset['Genotype'].unique()) < 3 or len(df_subset) < 10: # minimum dataset size
            trait_h2_info[f'{trt}_H2'] = np.nan
            trait_h2_info[f'{trt}_Var_G'] = np.nan
            trait_h2_info[f'{trt}_Var_E'] = np.nan
            pass_flag = False
            continue
            
        # calculate the repeats (r) per genotype  and use Harmonic mean (if there's repeated data)
        reps_per_geno = df_subset.groupby('Genotype').size()
        r = len(reps_per_geno) / np.sum(1.0 / reps_per_geno)
        
        try:
            # LMM Fitting: Value ~ 1 (Fixed effect: entire average), Random effect: Genotype
            model = smf.mixedlm("Value ~ 1", df_subset, groups=df_subset["Genotype"])
            result = model.fit(method='bfgs')
            
            # extract variance components (Random effect variance = variance_genetic, Residual variance = variance_error)
            var_G = result.cov_re.iloc[0, 0] # Intercept
            var_E = result.scale             # Residual 
            
            # correct negative variance 
            var_G = max(var_G, 0)
            
            # Heritability calculation
            if (var_G + var_E/r) > 0:
                h2 = var_G / (var_G + var_E / r)
            else:
                h2 = 0.0
                
            trait_h2_info[f'{trt}_H2'] = round(h2, 4)
            trait_h2_info[f'{trt}_Var_G'] = round(var_G, 4)
            trait_h2_info[f'{trt}_Var_E'] = round(var_E, 4)
            
            # if the heritability of trait under threshold (in Trt2)-> fail 
            if trt == 'Trt2' and h2 < H2_THRESHOLD:
                pass_flag = False
                
        except Exception as e:
            trait_h2_info[f'{trt}_H2'] = np.nan
            pass_flag = False

    trait_h2_info['Selection'] = 'Pass' if pass_flag else 'Exclude' 
    h2_results.append(trait_h2_info)

# Save the file and print summary
df_h2 = pd.DataFrame(h2_results)
save_path_h2 = os.path.join(base_dir, 'Phase2_Heritability_LMM.csv')
df_h2.to_csv(save_path_h2, index=False, encoding='utf-8-sig')

passed_traits = df_h2[df_h2['Selection'] == 'Pass']['Trait'].tolist()
print(f"Total traits evaluated: {len(traits)}")
print(f"Traits passed H2 threshold (>{H2_THRESHOLD} in Trt2): {len(passed_traits)}")
print("\n[Sample Output]")
print(df_h2[['Trait', 'Trt1_H2', 'Trt2_H2', 'Selection']].head(10))

In [ ]:
######################################
###  Phase 3. BLUP-based indexing  ###
######################################

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

base_dir = './'
raw_data_path = './HarvestData_2025.xlsx'
phase2_path = os.path.join(base_dir, 'Phase2_Heritability_LMM.csv')

print("Loading the data..")
df_raw = pd.read_excel(raw_data_path)
df_h2 = pd.read_csv(phase2_path)

# Filtering only passed traits from Heritability table
passed_traits = df_h2[df_h2['Selection'] == 'Pass']['Trait'].tolist()
blup_results = []

print(f"Extracting BLUPs and calculating DRI/STI for {len(passed_traits)} traits...")

# Extract BLUPs for each trait
for trait in passed_traits:
    blup_dict = {'Trt1': {}, 'Trt2': {}}
    
    for trt in ['Trt1', 'Trt2']:
        df_subset = df_raw[df_raw['Treatment'] == trt][['Genotype', trait]].dropna().copy()
        df_subset.columns = ['Genotype', 'Value']
        
        try:
            # Fit the LMM in conditions
            model = smf.mixedlm("Value ~ 1", df_subset, groups=df_subset["Genotype"])
            result = model.fit(method='bfgs')
            
            # Fixed effects (Intercept)
            intercept = result.params["Intercept"]
            # Random effects (Variance_Genotype)
            re = result.random_effects
            
            # BLUP_value = Fixed effects + Random effects
            for geno, re_series in re.items():
                blup_val = intercept + re_series.iloc[0]
                blup_dict[trt][geno] = blup_val
                
        except Exception as e:
            print(f"Error extracting BLUP for {trait} under {trt}: {e}")
            continue

    # Take the data existing only in pair for data integrity (Trt1,Trt2)
    common_genos = set(blup_dict['Trt1'].keys()).intersection(set(blup_dict['Trt2'].keys()))
    
    if not common_genos:
        continue
        
    # Average BLUP of trait_Trt1 (denominator of STI)
    Trt1_blups_list = [blup_dict['Trt1'][g] for g in common_genos]
    mean_ctrl_blup = np.mean(Trt1_blups_list)

    # BLUP-DRI, BLUP-STI Calculation for each genotype 
    for geno in common_genos:
        ctrl_val = blup_dict['Trt1'][geno]
        drt_val = blup_dict['Trt2'][geno]
        
        if ctrl_val != 0:
            dri = ((ctrl_val - drt_val) / ctrl_val) * 100
        else:
            dri = np.nan
            
        if mean_ctrl_blup != 0:
            sti = (ctrl_val * drt_val) / (mean_ctrl_blup ** 2)
        else:
            sti = np.nan
            
        blup_results.append({
            'Trait': trait,
            'Genotype': geno,
            'BLUP_Trt1': round(ctrl_val, 4),
            'BLUP_Trt2': round(drt_val, 4),
            'BLUP_DRI(%)': round(dri, 2),
            'BLUP_STI': round(sti, 4)
        })

# Save the file
df_phase3 = pd.DataFrame(blup_results)
save_path_phase3 = os.path.join(base_dir, 'Phase3_BLUP_Indices.csv')
df_phase3.to_csv(save_path_phase3, index=False, encoding='utf-8-sig')

print("-" * 50)
print("Phase 3 complete.")
print(f"Results saved to: {save_path_phase3}")
print("-" * 50)
print("\n[Sample Output]")
print(df_phase3.head(10))

In [ ]:
######################################
#######  Phase 4. Two-way LMM  #######
######################################

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter('ignore', ConvergenceWarning)

base_dir = './'
raw_data_path = './HarvestData.xlsx'
phase2_path = os.path.join(base_dir, 'Phase2_Heritability_LMM.csv')

print("Loading data..")
df_raw = pd.read_excel(raw_data_path)
df_h2 = pd.read_csv(phase2_path)

# Run only passed traits through phase 2
passed_traits = df_h2[df_h2['Selection'] == 'Pass']['Trait'].tolist()
phase4_results = []
print(f"Analyzing {len(passed_traits)} traits for treatment and GxE effects")

# LMM Fitting and extract variance components
for trait in passed_traits:
    df_clean = df_raw[['Genotype', 'Treatment', trait]].dropna().copy()
    df_clean.columns = ['Genotype', 'Treatment', 'Value']
    
    # Set the reference level in Trt1
    df_clean['Treatment'] = pd.Categorical(df_clean['Treatment'], categories=['Trt1', 'Trt2'])
    
    try:
        #################################################
        # [LMM Modeling structure]
        # - Fixed effect: Treatment
        # - Random effect: Genotype (Intercept)
        # - Random slope: Treatment (GxE Interaction)
        #################################################

        model = smf.mixedlm("Value ~ C(Treatment)", 
                            data=df_clean, 
                            groups=df_clean["Genotype"], 
                            re_formula="~C(Treatment)")
        result = model.fit(method='bfgs', maxiter=2000)
         
        p_treatment = result.pvalues.get("C(Treatment)[T.Trt2]", np.nan) # Fixed Effect: Does Trt2 treatment give significant change in average
        cov_matrix = result.cov_re  # Variance components
        var_G = max(cov_matrix.iloc[0, 0], 0)   
        var_GxE = max(cov_matrix.iloc[1, 1], 0) if cov_matrix.shape[0] > 1 else 0   
        var_Error = max(result.scale, 0)
        
        # calculate variance component proportions
        total_var = var_G + var_GxE + var_Error
        
        if total_var > 0:
            vcp_G = (var_G / total_var) * 100
            vcp_GxE = (var_GxE / total_var) * 100
            vcp_Error = (var_Error / total_var) * 100
        else:
            vcp_G, vcp_GxE, vcp_Error = 0, 0, 0
            
        phase4_results.append({
            'Trait': trait,
            'Treatment_p_value': p_treatment,
            'Var_Genotype(%)': round(vcp_G, 2),
            'Var_GxE(%)': round(vcp_GxE, 2),
            'Var_Error(%)': round(vcp_Error, 2),
            'Total_Variance': round(total_var, 4)
        })

    except Exception as e:
        print(f"Convergence/Fitting error for {trait}: {e}")
        continue

# Save the file
df_phase4 = pd.DataFrame(phase4_results)

# Order in high GxE_var
df_phase4 = df_phase4.sort_values(by='Var_GxE(%)', ascending=False)

save_path_phase4 = os.path.join(base_dir, 'Phase4_LMM_Effect_Structure.csv')
df_phase4.to_csv(save_path_phase4, index=False, encoding='utf-8-sig')

print("-" * 50)
print("Phase 4 LMM Analysis complete.")
print(f"Results saved to: {save_path_phase4}")
print("-" * 50)
print("\n[Top 5 Traits by GxE Interaction Variance Proportion]")
print(df_phase4[['Trait', 'Treatment_p_value', 'Var_Genotype(%)', 'Var_GxE(%)']].head())

In [ ]:
##########################################
#####  Phase 5.1. Scoring genotypes  #####
##########################################

import pandas as pd
import numpy as np
import scipy.stats as stats
import os

base_dir = './'
phase4_path = os.path.join(base_dir, 'Phase4_LMM_Effect_Structure.csv')
phase3_path = os.path.join(base_dir, 'Phase3_BLUP_Indices.csv')

df_p4 = pd.read_csv(phase4_path)
df_p3 = pd.read_csv(phase3_path)

# Trait selection (Treatment p < 0.05 AND Var_GxE >= 15.0)
top_traits_df = df_p4[(df_p4['Treatment_p_value'] < 0.05) & (df_p4['Var_GxE(%)'] >= 15.0)].copy()
selected_traits = top_traits_df['Trait'].tolist()

scoring_data = df_p3[df_p3['Trait'].isin(selected_traits)].copy()

print(f"Scoring {len(selected_traits)} selected traits...")

# Set the determination metric of traits (for particular traits use either DRI or STI)
def determine_metric(trait_name):
    # For structure data, apply DRI 
    dri_keywords = ['certain traits'] 
    if any(keyword in trait_name for keyword in dri_keywords):
        return 'BLUP_DRI(%)'
    else:
        # Rest data = apply STI 
        return 'BLUP_STI'

scoring_data['Metric_Used'] = scoring_data['Trait'].apply(determine_metric)

# 4. Z-score calculation
z_scores = []

for trait in selected_traits:
    trait_data = scoring_data[scoring_data['Trait'] == trait].copy()
    metric = trait_data['Metric_Used'].iloc[0]
    
    values = trait_data[metric].values
    
    # Z-score standardize
    z = stats.zscore(values, nan_policy='omit')
    
    # Direction Alignment
    if metric == 'BLUP_DRI(%)':
        z = z * -1      # since smaller DRI indicates excellent tolerance
        
    trait_data['Standardized_Z_Score'] = z
    z_scores.append(trait_data)

scoring_data_final = pd.concat(z_scores)

# Calculating composite tolerance score (CTS) per genotype using calculated z-score
genotype_scores = scoring_data_final.groupby('Genotype')['Standardized_Z_Score'].mean().reset_index()
genotype_scores.rename(columns={'Standardized_Z_Score': 'Composite_Tolerance_Score(CTS)'}, inplace=True)

# Ranking
genotype_scores = genotype_scores.sort_values(by='Composite_Tolerance_Score(CTS)', ascending=False).reset_index(drop=True)
genotype_scores['Rank'] = genotype_scores.index + 1


pivot_details = scoring_data_final.pivot(index='Genotype', columns='Trait', values='Standardized_Z_Score')

save_path_phase5 = os.path.join(base_dir, 'Phase5_Adaptive_Genotype_Tolerance_Ranking.csv')
genotype_scores.to_csv(save_path_phase5, index=False, encoding='utf-8-sig')

print("-" * 50)
print("Adaptive Scoring complete.")
print("\nTrt2-tolerant genotype Rank")
print(genotype_scores.to_string(index=False))

In [ ]:
#######################################
#####  Phase 5.2. Scoring traits  #####
#######################################

import pandas as pd
import matplotlib.pyplot as plt
import os

base_dir = './'
phase4_path = os.path.join(base_dir, 'Phase4_LMM_Effect_Structure.csv')

print("Loading data..")
df_p4 = pd.read_csv(phase4_path)

# Set strict trait selection Cut-off (Treatment_p-value < 0.05, GxE variance proportion >= 15%)
P_VALUE_THRESHOLD = 0.05
GXE_THRESHOLD = 15.0

# Satisfied trait filtering
selected_traits_df = df_p4[
    (df_p4['Treatment_p_value'] < P_VALUE_THRESHOLD) & 
    (df_p4['Var_GxE(%)'] >= GXE_THRESHOLD)
].copy()

# Trait scoring and ranking via Var_GxE(%)
selected_traits_df = selected_traits_df.sort_values(by='Var_GxE(%)', ascending=False).reset_index(drop=True)
selected_traits_df['Trait_Rank'] = selected_traits_df.index + 1

save_path_traits = os.path.join(base_dir, 'Phase4_Selected_Traits_Ranking.csv')
selected_traits_df.to_csv(save_path_traits, index=False, encoding='utf-8-sig')

print("-" * 50)
print(f"Trait Selection Complete: {len(selected_traits_df)} traits passed the rigorous cut-off.")
print(f"Results saved to: {save_path_traits}")
print("-" * 50)
print("\n[Top Selected Traits Ranking]")
print(selected_traits_df[['Trait_Rank', 'Trait', 'Treatment_p_value', 'Var_GxE(%)', 'Var_Genotype(%)']].head(10).to_string(index=False))




###---------- Visualization with bar stack chart------------###

try:
    plot_df = selected_traits_df.copy()
    plot_df.set_index('Trait', inplace=True)
    plot_df = plot_df[['Var_GxE(%)', 'Var_Genotype(%)', 'Var_Error(%)']]
    
    colors = ['red', 'blue', 'grey']
    
    fig_height = max(8, len(plot_df) * 0.5)
    fig, ax = plt.subplots(figsize=(10, fig_height))
    
    plot_df.plot(kind='barh', stacked=True, color=colors, ax=ax, edgecolor='black')
    
    ax.set_title('Variance Component Proportions of Selected Traits', fontsize=14, fontweight='bold')
    ax.set_xlabel('Proportion of Total Variance (%)', fontsize=12)
    ax.set_ylabel('Traits', fontsize=12)
    ax.legend(['GxE Interaction', 'Genotype (Baseline)', 'Error (Noise)'], 
              loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)
    
    plt.gca().invert_yaxis() # 1st rank trait comes upper
    plt.tight_layout()
    
    plot_save_path = os.path.join(base_dir, 'Phase4_Selected_All_Traits_VCP_Plot.png')
    plt.savefig(plot_save_path, dpi=300)
    print(f"\n[Plot saved] Effect structure visualization saved to: {plot_save_path}")

except Exception as e:
    print(f"\nCould not generate plot: {e}")

In [ ]:
###################################################################
#####  Phase 5.2. Scoring more traits for correlation matrix  #####
###################################################################

import pandas as pd
import matplotlib.pyplot as plt
import os

base_dir = './'
phase4_path = os.path.join(base_dir, 'Phase4_LMM_Effect_Structure.csv')

print("Loading data..")
df_p4 = pd.read_csv(phase4_path)

# Set strict trait selection Cut-off (Treatment_p-value < 0.05, GxE variance proportion >= 15%)
P_VALUE_THRESHOLD = 0.05
GXE_THRESHOLD = 10.0

# Satisfied trait filtering
selected_traits_df = df_p4[
    (df_p4['Treatment_p_value'] < P_VALUE_THRESHOLD) & 
    (df_p4['Var_GxE(%)'] >= GXE_THRESHOLD)
].copy()

# Trait scoring and ranking via Var_GxE(%)
selected_traits_df = selected_traits_df.sort_values(by='Var_GxE(%)', ascending=False).reset_index(drop=True)
selected_traits_df['Trait_Rank'] = selected_traits_df.index + 1

save_path_traits = os.path.join(base_dir, 'Phase4.2_Selected_Traits_Ranking_correlation.csv')
selected_traits_df.to_csv(save_path_traits, index=False, encoding='utf-8-sig')

print("-" * 50)
print(f"Trait Selection Complete: {len(selected_traits_df)} traits passed the rigorous cut-off.")
print(f"Results saved to: {save_path_traits}")
print("-" * 50)
print("\n[Top Selected Traits Ranking]")
print(selected_traits_df[['Trait_Rank', 'Trait', 'Treatment_p_value', 'Var_GxE(%)', 'Var_Genotype(%)']].head(10).to_string(index=False))


Loading data..
--------------------------------------------------
Trait Selection Complete: 17 traits passed the rigorous cut-off.
Results saved to: /mnt/c/Users/채은비/Desktop/Master_thesis/03_Results/0301_statistics/Phase4.2_Selected_Traits_Ranking_correlation.csv
--------------------------------------------------

[Top Selected Traits Ranking]
 Trait_Rank                      Trait  Treatment_p_value  Var_GxE(%)  Var_Genotype(%)
          1                 d13C_Seeds       1.673562e-10       43.32            37.79
          2  RipePaniclesWeight_Grains       1.306379e-03       35.48            57.79
          3              Weight_Grains       1.724397e-03       34.54            59.12
          4  RipePaniclesNumber_Grains       1.693677e-03       33.81            58.97
          5   RipePaniclesWeight_dried       6.717405e-04       32.33            60.99
          6              Number_Grains       1.353060e-03       31.10            61.74
          7 RipePaniclesWeight_harvest       